In [0]:
%sql
USE CATALOG ipl_2024_project;
USE SCHEMA pyspark;

#### Reference diag -->

![image_1781069851421.png](./image_1781069851421.png "image_1781069851421.png")

![image_1781069913633.png](./image_1781069913633.png "image_1781069913633.png")

### Batting Scorecard (Batsman Level)

#### Information Required
###### 1. Team Name
###### 2. Batsman Name
###### 3. Runs
###### 4. Balls
###### 5. 4s
###### 6. 6s
###### 7. Strike Rate

In [0]:
team = spark.read.table("team")
innings = spark.read.table("innings")
player = spark.read.table("player")
score_by_ball = spark.read.table("score_by_ball")

In [0]:
from pyspark.sql.functions import sum, col, count, concat, lit, when, min, format_number
i = innings.alias("i")
t = team.alias("t")
s = score_by_ball.alias("s")
p = player.alias("p")

i.join(
    t,
    i.batting_team_id == t.team_id,
    "inner"
).join(
    s,
    (s.match_id == i.match_id) & (s.innings_no == i.innings_no),
    "inner"
).join(
    p,
    p.player_id == s.striker_id,
    "inner"
).filter(
    s.wides.isNull() & s.noballs.isNull()
).groupBy(
    i.match_id,
    i.innings_no,
    t.team_name.alias("batting_team_name"),
    p.player_name.alias("batsman_name")
).agg(
    sum(s.runs_off_bat).alias("total_runs"),
    count(s.ball_no).alias("total_balls"),
    sum(
        when(s.runs_off_bat == 4, 1).otherwise(0)
    ).alias("no_of_fours"),
    sum(
        when(s.runs_off_bat == 6, 1).otherwise(0)
    ).alias("no_of_sixes"),
    format_number(
        (sum(s.runs_off_bat) / count(s.ball_no)) * 100, 
        2
    ).alias("strike_rate")
).orderBy(
    i.match_id,
    i.innings_no,
    min(s.ball_no)
).display()

### Batting Scorecard (Team Level)

#### Information Required
###### 1. Team Name
###### 2. extras
###### 3. noballs
###### 4. wides
###### 5. byes
###### 6. legbyes
###### 7. penalities

In [0]:
from pyspark.sql.functions import sum, col, count, concat, lit, when, min, format_number, coalesce
i = innings.alias("i")
t = team.alias("t")
s = score_by_ball.alias("s")
p = player.alias("p")

i.join(
    t,
    i.batting_team_id == t.team_id,
    "inner"
).join(
    s,
    (s.match_id == i.match_id) & (s.innings_no == i.innings_no),
    "inner"
).groupBy(
    i.match_id,
    i.innings_no,
    t.team_name.alias("batting_team_name")
).agg(
    sum(s.extras).alias("total_extras"),
    # sum(
    #     when(s.noballs.isNull(),0).otherwise(s.noballs)
    # ).alias("total_noballs"),
    sum(coalesce(s.noballs,lit(0))).alias("total_noballs"),
    sum(s.wides).alias("total_wides"),
    sum(s.byes).alias("total_byes"),
    sum(s.legbyes).alias("total_legbyes"),
    sum(coalesce(s.penalty,lit(0))).alias("total_penalty")
).display()
